In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS final_project.gold")

In [0]:
train = spark.table("final_project.silver.train")
test  = spark.table("final_project.silver.test")


## Time-based features



Here we derive general time features from pickup_datetime, such as:

- time (copy of pickup timestamp)
- year, month, day
- hour
- day_of_week (1 = Monday, ..., 7 = Sunday)

These features help the model capture temporal patterns in taxi demand and pricing.


In [0]:
def add_time_features(df):
    df = df.withColumn("time", F.col("pickup_datetime"))
    df = df.withColumn("year",  F.year("pickup_datetime"))
    df = df.withColumn("month", F.month("pickup_datetime"))
    df = df.withColumn("day",   F.dayofmonth("pickup_datetime"))
    df = df.withColumn("hour",  F.hour("pickup_datetime"))
    df = df.withColumn("day_of_week", F.date_format("pickup_datetime", "u").cast("int"))
    return df

train_fe = add_time_features(train)
test_fe  = add_time_features(test)

## Calendar encodings (month & weekday flags)


Next, we one-hot encode:

- Months: jan, feb, ..., dec
- Weekdays: mon, tue, wed...

These binary indicators allow tree-based models and linear models to learn seasonality and weekly patterns more easily.

In [0]:
months = [
    ("jan", 1), ("feb", 2), ("mar", 3), ("apr", 4),
    ("may", 5), ("june", 6), ("july", 7), ("aug", 8),
    ("sep", 9), ("oct", 10), ("nov", 11), ("dec", 12)
]

for name, m in months:
    train_fe = train_fe.withColumn(name, (F.col("month") == m).cast("int"))
    test_fe  = test_fe.withColumn(name, (F.col("month") == m).cast("int"))

In [0]:

days = [
    ("mon", 1), ("tue", 2), ("wed", 3), ("thu", 4),
    ("fri", 5), ("sat", 6), ("sun", 7)
]

for name, d in days:
    train_fe = train_fe.withColumn(name, (F.col("day_of_week") == d).cast("int"))
    test_fe  = test_fe.withColumn(name, (F.col("day_of_week") == d).cast("int"))


## Travel vector features and local outlier removal


We compute basic geospatial deltas:

- abs_diff_longitude = |dropoff_longitude - pickup_longitude|
- abs_diff_latitude  = |dropoff_latitude - pickup_latitude|

Then, for the train set only, we remove extreme trips where:

- abs_diff_longitude >= 5.0 or
- abs_diff_latitude >= 5.0

These rules further reduce unrealistic trips that may harm model training.

In [0]:
def add_travel_vector(df):
    df = df.withColumn(
        "abs_diff_longitude",
        F.abs(F.col("dropoff_longitude") - F.col("pickup_longitude"))
    )
    df = df.withColumn(
        "abs_diff_latitude",
        F.abs(F.col("dropoff_latitude") - F.col("pickup_latitude"))
    )
    return df

train_fe = add_travel_vector(train_fe)
test_fe  = add_travel_vector(test_fe)


In [0]:
train_fe = train_fe.filter(
    (F.col("abs_diff_longitude") < 5.0) &
    (F.col("abs_diff_latitude") < 5.0)
)

## Haversine distance between pickup and dropoff



We compute a more realistic trip distance feature:

- hs_dist – great-circle distance (in km) between pickup and dropoff coordinates using the Haversine formula.

This distance is a key predictor of fare amount and improves the model compared to raw coordinate differences alone.


In [0]:
def haversine_cols(lon1_col, lat1_col, lon2_col, lat2_col):
    """
    Returns a Column with haversine distance (km) between two points.
    """
    R = 6378.0

    lon1 = F.radians(lon1_col)
    lat1 = F.radians(lat1_col)
    lon2 = F.radians(lon2_col)
    lat2 = F.radians(lat2_col)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (F.sin(dlat / 2) ** 2 +
         F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2)
    c = 2 * F.asin(F.sqrt(a))
    return R * c

train_fe = train_fe.withColumn(
    "hs_dist",
    haversine_cols(
        F.col("pickup_longitude"), F.col("pickup_latitude"),
        F.col("dropoff_longitude"), F.col("dropoff_latitude"),
    )
)

In [0]:
test_fe = test_fe.withColumn(
    "hs_dist",
    haversine_cols(
        F.col("pickup_longitude"), F.col("pickup_latitude"),
        F.col("dropoff_longitude"), F.col("dropoff_latitude"),
    )
)

## Distances to airports and NYC landmarks

Using a predefined set of important locations (e.g., JFK, LGA, EWR, WTC, Statue of Liberty), we add features like:

- jfk_drop_distance
- lga_drop_distance
- ewr_drop_distance
- ... (and other landmark _drop_distance columns)

Each feature measures the Haversine distance from the dropoff point to that landmark.  
This helps the model capture typical flat-fare regions (e.g., airport trips) and popular tourist/downtown areas.

In [0]:
sights = {
    "jfk": (-73.7781, 40.6413),
    "lga": (-73.8740, 40.7769),
    "ewr": (-74.1745, 40.6895),
    "met": (-73.9632, 40.7794),
    "wtc": (-74.0099, 40.7126),
    "sol": (-74.0445, 40.6892),
    "nyc": (-74.0063889, 40.7141667),
    "tms": (-73.9854406, 40.7581047),
    "plz": (-73.9750593, 40.7651662),
    "hbk": (-74.0282202, 40.7356908),
    "brb": (-73.9741970, 40.5882819),
}

for name, (lon, lat) in sights.items():
    col_name = f"{name}_drop_distance"
    train_fe = train_fe.withColumn(
        col_name,
        haversine_cols(
            F.lit(lon), F.lit(lat),
            F.col("dropoff_longitude"), F.col("dropoff_latitude")
        ),
    )
    test_fe = test_fe.withColumn(
        col_name,
        haversine_cols(
            F.lit(lon), F.lit(lat),
            F.col("dropoff_longitude"), F.col("dropoff_latitude")
        ),
    )


## Pickup and dropoff regions (KMeans clustering)



We cluster coordinates using KMeans:

- pickup_region – cluster label based on pickup longitude/latitude
- dropoff_region – cluster label based on dropoff longitude/latitude

These region IDs act as compact categorical features that capture neighborhood-level effects and local pricing patterns.

In [0]:
clusters = 20

pickup_assembler = VectorAssembler(
    inputCols=["pickup_longitude", "pickup_latitude"],
    outputCol="pickup_features",
)

train_pickup = pickup_assembler.transform(train_fe)

kmeans_pickup = KMeans(
    k=clusters,
    seed=42,
    featuresCol="pickup_features",
    predictionCol="pickup_region",
)

pickup_model = kmeans_pickup.fit(train_pickup)

train_fe = pickup_model.transform(train_pickup)

test_pickup = pickup_assembler.transform(test_fe)
test_fe = pickup_model.transform(test_pickup)

dropoff_assembler = VectorAssembler(
    inputCols=["dropoff_longitude", "dropoff_latitude"],
    outputCol="dropoff_features",
)

train_dropoff = dropoff_assembler.transform(train_fe)

kmeans_dropoff = KMeans(
    k=clusters,
    seed=42,
    featuresCol="dropoff_features",
    predictionCol="dropoff_region",
)

dropoff_model = kmeans_dropoff.fit(train_dropoff)

train_fe = dropoff_model.transform(train_dropoff)

test_dropoff = dropoff_assembler.transform(test_fe)
test_fe = dropoff_model.transform(test_dropoff)

train_fe = train_fe.drop("pickup_features", "dropoff_features")
test_fe  = test_fe.drop("pickup_features", "dropoff_features")


## Weather enrichment and binary weather flags

We load historical NYC weather data (pre-downloaded into a Volume), derive calendar fields, and join it to trips on:

- year, month, day

From the joined weather columns (Events, Mean_Wind_SpeedMPH, Mean_TemperatureF) we create:

- bad_weather  – 1 if there was a recorded weather event
- strong_wind  – 1 if mean wind speed ≥ 18 mph
- cold_weather – 1 if mean temperature ≤ 40°F

These features help the model capture fare and trip pattern changes due to adverse weather.

In [0]:
weather_path = 'dbfs:/Volumes/final_project/default/files/weather_df.csv'

orig_weather_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(weather_path)
)

In [0]:
for old in orig_weather_df.columns:
    new = old.replace(".", "_")
    orig_weather_df = orig_weather_df.withColumnRenamed(old, new)

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


orig_weather_df = orig_weather_df.withColumn(
    "time", F.to_date(F.col("Date"), "yyyy-MM-dd")
)

orig_weather_df = orig_weather_df.withColumn("time", F.to_date(F.col("Date"), "yyyy-MM-dd"))


weather_df = (
    orig_weather_df
    .withColumn("year",  F.year(F.col("time")))
    .withColumn("month", F.month(F.col("time")))
    .withColumn("day",   F.dayofmonth(F.col("time")))
)

weather_df = weather_df.filter(F.col("year") >= 2009)

weather_df = weather_df.select(
    "year",
    "month",
    "day",
    "Events",
    "Mean_Wind_SpeedMPH",
    "Mean_TemperatureF"
)

train_fe = train_fe.join(weather_df, on=["year", "month", "day"], how="left")
test_fe  = test_fe.join(weather_df, on=["year", "month", "day"], how="left")

train_fe = (
    train_fe
    .withColumn("bad_weather",  F.when(F.col("Events").isNotNull(), 1).otherwise(0))
    .withColumn("strong_wind",  F.when(F.col("Mean_Wind_SpeedMPH") >= 18, 1).otherwise(0))
    .withColumn("cold_weather", F.when(F.col("Mean_TemperatureF") <= 40, 1).otherwise(0))
)

test_fe = (
    test_fe
    .withColumn("bad_weather",  F.when(F.col("Events").isNotNull(), 1).otherwise(0))
    .withColumn("strong_wind",  F.when(F.col("Mean_Wind_SpeedMPH") >= 18, 1).otherwise(0))
    .withColumn("cold_weather", F.when(F.col("Mean_TemperatureF") <= 40, 1).otherwise(0))
)

## Persist feature-engineered gold tables


We write the final feature-enriched DataFrames as Delta tables:

- final_project.gold.train_features
- final_project.gold.test_features

These gold tables are the model-ready input for training and evaluation in the next step of the project.


In [0]:
train_fe = train_fe.drop("_c0", "time")
test_fe = test_fe.drop("time")

In [0]:
(
    train_fe.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("final_project.gold.train")
)

(
    test_fe.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("final_project.gold.test")
)

print("Created final_project.gold.train and final_project.gold.test")

In [0]:
train_df = spark.table("final_project.gold.train")
test_df = spark.table("final_project.gold.test")

display(train_df)
display(test_df)